[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/08_data_vis/08_2_Comparing_Categories.ipynb)

# 08.2: Comparing Categories

Notebook 08.1 used a bar chart to count penguins. A **bar chart** draws one bar per category, and the length of the bar stands for a number. That simple idea leaves several decisions to you: what order to put the bars in, where the axis starts, whether to show counts or rates, and whether one number per category is even enough. This notebook works through those decisions.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

We use a table of college majors, where one row is one major, and a table of FBI crime data, where one row is one state in one year. `Median` in the majors table is the median earnings of recent graduates of that major, in dollars.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent_Rate", "Totals.Violent.All": "Violent_Total"})
crime = crime[["State", "Year", "Population", "Violent_Rate", "Violent_Total"]]
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
print(majors.shape)
print(places_2012.shape)

There are 173 majors, and `places_2012` holds the 50 states and the District of Columbia for 2012. We leave out the row for the United States as a whole, since it is not a place to compare with the others.

## Counts and calculated numbers

`sns.countplot()` counts rows for you, as you saw in 08.1. To show any other number for each category, the usual approach is to calculate it yourself with `.groupby()` and `.reset_index()` from notebook 07.4, and then pass the small table to `sns.barplot()`. Calculating first makes it clear exactly what the bars show.

In [ ]:
pay = majors.groupby("Major_category")["Median"].mean().reset_index()
print(len(pay))
ax = sns.barplot(data=pay, x="Major_category", y="Median")
ax.set_title("Median earnings by category of major")
ax.set_xlabel("Category of major")
ax.set_ylabel("Mean of the median earnings (dollars)")
plt.show()

There are 16 categories, and this chart has two problems. The labels run on top of each other, and the bars are in alphabetical order, which has nothing to do with the question. You can see that one bar is tallest, but you cannot read which one it is.

## Decision 1: order and direction

The fix has two parts. Put the category on the y axis, which gives long labels room to be read, and sort the table so that the order of the bars means something. Seaborn draws the bars in the order of the rows.

In [ ]:
pay_sorted = pay.sort_values("Median", ascending=False)
ax = sns.barplot(data=pay_sorted, x="Median", y="Major_category")
ax.set_title("Median earnings by category of major")
ax.set_xlabel("Mean of the median earnings (dollars)")
ax.set_ylabel("")
plt.show()

Now the chart answers a question at a glance. Engineering is far ahead at about 57383 dollars, and the other categories run from Business at about 43538 down to Psychology & Social Work at about 30100. We could read that ranking only because the bars are sorted.

Sort by value when the categories have no natural order. When they do have one, such as small, medium and large, keep that order, because a reader expects it.

If there are too many categories to read even on a horizontal chart, keep only the ones that matter to your question and say so in the title, as in the next chart. You will sometimes see long labels tilted instead, using `plt.xticks(rotation=45, ha="right")` before `plt.show()`. That works, but tilted text is harder to read than level text, so try a horizontal chart first.

## Decision 2: start the axis at zero

Readers judge a bar by its length, so the length has to be proportional to the number. That only holds if the axis starts at zero. Here are the rates for Tennessee and Nevada, the second and third highest in the data, with an axis that starts at 600. `ax.set_ylim()` sets the range of the y axis.

In [ ]:
top_ten = places_2012.sort_values("Violent_Rate", ascending=False).head(10)
tn_nv = top_ten[top_ten["State"].isin(["Tennessee", "Nevada"])]
ax = sns.barplot(data=tn_nv, x="State", y="Violent_Rate")
ax.set_title("A y axis that starts at 600")
ax.set_xlabel("")
ax.set_ylabel("Violent crimes per 100,000 people")
ax.set_ylim(600, 650)
plt.show()

Tennessee's bar looks nearly six times as tall as Nevada's, because only the 43.6 and the 7.6 above 600 are drawn. The rates are 643.6 and 607.6, so Tennessee's is only about 6 percent higher. Here is the same data with the axis starting at zero, which is what seaborn does for a bar chart unless you change it.

In [ ]:
ax = sns.barplot(data=tn_nv, x="State", y="Violent_Rate")
ax.set_title("The same data with a y axis that starts at zero")
ax.set_xlabel("")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

The bars are now close to the same height, which matches the numbers. When you read someone else's bar chart, check where its axis starts before you trust your first impression.

Zero is not always a meaningful place to start, though. Life expectancy is a good example, since nobody compares countries by how far they are above an age of 0. In a bar chart starting at zero, the differences among countries nearly disappear, and an axis starting at 70 would distort the bars. A **dot plot** avoids the problem. A dot marks a value by its position rather than its length, so its axis does not need to include zero.

In [ ]:
health = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_healthexp.csv")
health_2020 = health[health["Year"] == 2020].sort_values("Life_Expectancy", ascending=False)
ax = sns.scatterplot(data=health_2020, x="Life_Expectancy", y="Country", s=100)
ax.set_title("Life expectancy in 2020")
ax.set_xlabel("Life expectancy (years)")
ax.set_ylabel("")
plt.show()

Japan is highest at 84.7 years and the USA is lowest at 77.0, a gap of 7.7 years that matters a great deal even though it is small compared with the distance from zero. The axis here runs only from about 77 to 85, which is fine for dots but would not be for bars.

## Decision 3: counts or rates?

A count often answers a different question from the one you meant to ask. Which states had the most violent crime in 2012?

In [ ]:
top_total = places_2012.sort_values("Violent_Total", ascending=False).head(10)
ax = sns.barplot(data=top_total, x="Violent_Total", y="State")
ax.set_title("Ten highest numbers of violent crimes, 2012")
ax.set_xlabel("Number of violent crimes")
ax.set_ylabel("")
plt.show()

California is first with 160944 crimes, followed by Texas with 106476 and Florida with 94087. These are also among the largest states, so this chart is partly a chart of population. A **rate** divides by population, which here gives crimes per 100,000 people. Now compare the same question using rates.

In [ ]:
ax = sns.barplot(data=top_ten, x="Violent_Rate", y="State")
ax.set_title("Ten highest violent crime rates, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("")
plt.show()

The ranking changes completely. The District of Columbia is first at 1243.7, with only 7864 crimes, and California is not in the top ten at all. Neither chart is wrong, because they answer different questions. The counts tell you where the most crimes happened, which could matter for planning police budgets. The rates tell you how common crime is relative to the number of people, which is what you usually want when you compare places of different sizes.

## Decision 4: is one number enough?

A bar of means reduces a whole group to one number, and groups with very different shapes can have similar means. Here are the weekly grosses of Broadway shows by type, where one row is one show in one week.

If you give `sns.barplot()` a table with many rows per category, it calculates the mean for you and adds a thin line showing the uncertainty in that mean. This is a shortcut for the `.groupby()` step above.

In [ ]:
broadway = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
ax = sns.barplot(data=broadway, x="Type", y="Gross")
ax.set_title("Mean weekly gross by type of show")
ax.set_xlabel("Type of show")
ax.set_ylabel("Mean weekly gross (dollars)")
plt.show()
broadway.groupby("Type")["Gross"].agg(["count", "mean", "median"])

The chart suggests that specials earn a little more than plays, at about 365322 dollars against about 325718. The table shows something the bars cannot: the median for specials, 240084, is below the median for plays, 262396. A few very large weeks pull up the mean for specials, and there are only 339 rows of them compared with 8406 for plays. So the answer to "do specials earn more than plays?" depends on which number you use. When the shape of a group matters, a bar chart is the wrong tool, and notebook 08.4 shows what to use instead.

## Decision 5: two categories at once

Sometimes the question involves two categories. We use the Titanic passenger list, where one row is one passenger. Did survival differ across the three passenger classes? `.map()` replaces each value using a dictionary, so that the labels are readable.

`hue=` draws a separate bar in a separate color for each value of a second column. This is called a **grouped bar chart**.

In [ ]:
titanic = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv")
titanic["Outcome"] = titanic["Survived"].map({0: "Died", 1: "Survived"})
ax = sns.countplot(data=titanic, x="Pclass", hue="Outcome", hue_order=["Died", "Survived"])
ax.set_title("Titanic passengers by class and outcome")
ax.set_xlabel("Passenger class")
ax.set_ylabel("Number of passengers")
plt.show()

The counts show that third class was by far the largest group, with 487 of the 887 passengers, and that most of them died. It is harder to compare the survival rates, because the groups differ so much in size. A **standardized bar chart** makes every bar the same height and shows the share in each part, the same idea as `normalize="index"` in a crosstab. `sns.histplot()` can draw one with `multiple="fill"`. `discrete=True` tells it that each value of `Pclass` is its own bar, and `shrink=` leaves gaps between them. `sns.move_legend()` puts the legend outside the chart, where it does not cover the bars.

In [ ]:
ax = sns.histplot(data=titanic, x="Pclass", hue="Outcome", multiple="fill", discrete=True, shrink=0.8)
ax.set_title("Share of each class that survived")
ax.set_xlabel("Passenger class")
ax.set_ylabel("Share of passengers")
ax.set_xticks([1, 2, 3])
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.show()

Now the comparison is direct. About 63 percent of first class survived, about 47 percent of second class and about 24 percent of third class. The cost is that the chart no longer shows how many people each bar represents, so a reader cannot tell that third class was more than twice the size of first class. Neither chart replaces the other: use the counts if size matters and the shares if you are comparing rates.

Put the category you are explaining with on the x axis. Here class is on the x axis, so each bar answers "of the people in this class, how many survived?" Swapping the two columns would answer a different question: "of the people who survived, how many were in each class?" This is the same distinction as between `normalize="index"` and `normalize="columns"` in notebook 07.4.

You will also see **stacked bars** that are not standardized, where the parts sit on top of each other. Only the bottom part starts from a common baseline, which makes the upper parts hard to compare, and stacking only makes sense if the total means something. Stacking counts of people gives a meaningful total, but stacking median earnings would not.

## What about pie charts?

A pie chart shows how a whole divides into parts, using the angle and the area of each slice. Research on how people read charts has found that people judge position most accurately, then length, then angle and area, with color least accurate. That is why a bar chart usually makes a comparison easier than a pie chart. A pie can work when there are only a few slices and the point is a simple fraction, such as about half. It works poorly with many slices. `plt.pie()` from matplotlib draws one, since seaborn does not have a pie chart.

In [ ]:
category_counts = majors["Major_category"].value_counts()
plt.pie(category_counts.values, labels=category_counts.index)
plt.title("Majors by category")
plt.show()

You can see that Engineering is the largest slice, with 29 majors, but it is very hard to rank the middle slices or to compare slices that are not next to each other, such as Education with 16 and Business with 13. A sorted horizontal bar chart of the same counts would make all of those comparisons easy.

## Figure size

Charts have a default size. If you need a different one, call `plt.figure(figsize=(10, 6))` before the plotting call, where the numbers are the width and the height. This is useful for a horizontal bar chart with many categories.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Property.All": "Property_Rate", "Totals.Property.All": "Property_Total"})
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
two_states = places_2012[places_2012["State"].isin(["Illinois", "Michigan"])]
titanic = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv")
titanic["Outcome"] = titanic["Survived"].map({0: "Died", 1: "Survived"})
share_want = majors.groupby("Major_category")["ShareWomen"].mean().reset_index().sort_values("ShareWomen", ascending=False)
top_rates_want = places_2012.sort_values("Property_Rate", ascending=False).head(5)["Property_Rate"].to_list()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
Create a DataFrame named `share_table` with `Major_category` and `ShareWomen` as ordinary columns, where `ShareWomen` is the mean for each category, sorted from highest to lowest.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "share_table", share_want, "Group, take the mean of ShareWomen, use .reset_index(), then .sort_values() with ascending=False.")

### Problem 2
Use `share_table` to make a horizontal bar chart with the highest category at the top, and store it in `share_bars`. Give it a title and an x axis label.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check_chart(2, "share_bars", lambda c: np.allclose([p.get_width() for p in c.patches if p.get_width() > 0], share_want["ShareWomen"]), "Put ShareWomen on x and Major_category on y, using the sorted table.", y_label=False)

### Problem 3
The cell below already has code that compares the property crime rates of Illinois and Michigan. Run it and look at the chart. It makes the gap look large, but the rates are 2578.7 and 2530.5, so Illinois is only about 2 percent higher. Fix the code so that the y axis starts at zero, and change the title so that it no longer claims a large difference. Keep the chart stored in `fair_bars`.

In [ ]:
# Problem 3
fair_bars = sns.barplot(data=two_states, x="State", y="Property_Rate")
fair_bars.set_title("Illinois has far more property crime than Michigan, 2012")
fair_bars.set_xlabel("State")
fair_bars.set_ylabel("Property crimes per 100,000 people")
fair_bars.set_ylim(2500, 2600)
plt.show()

In [ ]:
#@title Check Problem 3
check_chart(3, "fair_bars", lambda c: c.get_ylim()[0] == 0 and "far more" not in c.get_title().lower() and np.allclose(sorted(p.get_height() for p in c.patches if p.get_height() > 0), [2530.5, 2578.7]), "Remove the set_ylim() line or start the axis at 0, and write a title that does not say far more.")

### Problem 4
Make a horizontal bar chart of the five places in `places_2012` with the highest `Property_Rate`, ordered from highest to lowest, and store it in `rate_bars`. Give it a title and an x axis label.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check_chart(4, "rate_bars", lambda c: np.allclose([p.get_width() for p in c.patches if p.get_width() > 0], top_rates_want), "Sort by Property_Rate, take the first five, and put State on y.", y_label=False)

### Problem 5
You want to know where a person is most likely to be a victim of property crime. Would you compare states using totals or rates? Store your answer in a variable named `choice_5`, as one of `"totals"` or `"rates"`.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "choice_5", "rates", "A total depends on how many people live in the state.")

### Problem 6
Using `titanic`, make a standardized bar chart with `Sex` on the x axis and `Outcome` as the hue, and store it in `sex_shares`. Give it a title and label both axes.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check_chart(6, "sex_shares", lambda c: len([p for p in c.patches if p.get_height() > 0]) == 4 and c.get_ylim()[1] <= 1.05, "Use sns.histplot() with multiple=\"fill\", discrete=True and shrink=0.8.")

### Problem 7
You want to compare the 16 categories of majors by how many majors each has. Would you use a pie chart or a bar chart? Store your answer in a variable named `choice_7`, as one of `"pie"` or `"bar"`.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "choice_7", "bar", "Sixteen slices are hard to compare, and people judge length more accurately than angle.")

## What's next

You can now build a bar chart that is sorted, starts at zero and shows the right number. The last decision showed the limit of a bar chart: it gives one number per category and hides everything else about the group. To see a whole numeric column, you need a different kind of chart. Notebook 08.3 covers histograms, and the choice that affects them most: the size of the bins.